# 02 — Baseline Convolutional Autoencoder

This notebook:
1. Loads and inspects the dataset / dataloaders
2. Defines and inspects the `ConvAutoencoder` model
3. Runs a forward-pass sanity check
4. Trains the model on normal images (MAE / L1 loss)
5. Evaluates reconstruction quality on the validation set


## 1. Imports & Reproducibility

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import matplotlib.pyplot as plt
import numpy as np

from utils import set_seed, CHECKPOINTS_DIR
from dataset import get_dataloaders
from models import ConvAutoencoder

set_seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 2. Dataset & DataLoaders

In [ ]:
BATCH_SIZE = 16

train_loader, val_loader, test_loader = get_dataloaders(
    batch_size=BATCH_SIZE, val_split=0.15, seed=42
)

n_train = len(train_loader.dataset)
n_val   = len(val_loader.dataset)
n_test  = len(test_loader.dataset)

print(f'Train samples : {n_train}')
print(f'Val   samples : {n_val}')
print(f'Test  samples : {n_test}')
print(f'Train + Val   : {n_train + n_val}  (expected 224)')

In [ ]:
# Inspect a batch shape
sample_batch = next(iter(train_loader))
print(f'Batch shape : {sample_batch.shape}')  # [16, 3, 256, 256]
print(f'Dtype       : {sample_batch.dtype}')
print(f'Min / Max   : {sample_batch.min():.3f} / {sample_batch.max():.3f}')

In [ ]:
# Visualise a grid of training samples (un-normalised for display)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

def denorm(t):
    """Reverse ImageNet normalisation for display."""
    img = t.permute(1, 2, 0).numpy() * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(img, 0, 1)

fig, axes = plt.subplots(2, 8, figsize=(18, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(denorm(sample_batch[i]))
    ax.axis('off')
fig.suptitle('Sample training batch (16 normal images, 256x256)', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Model Definition & Inspection

In [ ]:
model = ConvAutoencoder().to(DEVICE)
print(model)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters    : {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 4. Forward-Pass Sanity Check

In [ ]:
model.eval()
with torch.no_grad():
    x     = sample_batch.to(DEVICE)
    x_hat = model(x)

assert x_hat.shape == x.shape, f'Shape mismatch: {x_hat.shape} != {x.shape}'
print(f'Input  shape : {x.shape}')
print(f'Output shape : {x_hat.shape}')
print('Sanity check passed.')

## 5. Training

Train the autoencoder on normal images only using MAE (L1) loss.

In [ ]:
import torch.optim as optim

NUM_EPOCHS = 50
LR         = 1e-3

criterion = torch.nn.L1Loss()  # MAE loss
optimizer = optim.Adam(model.parameters(), lr=LR)

train_losses = []
val_losses   = []

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    running_loss = 0.0
    for imgs in train_loader:
        imgs = imgs.to(DEVICE)
        optimizer.zero_grad()
        recon = model(imgs)
        loss  = criterion(recon, imgs)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    train_loss = running_loss / len(train_loader.dataset)

    # --- Validate ---
    model.eval()
    running_val = 0.0
    with torch.no_grad():
        for imgs in val_loader:
            imgs  = imgs.to(DEVICE)
            recon = model(imgs)
            running_val += criterion(recon, imgs).item() * imgs.size(0)
    val_loss = running_val / len(val_loader.dataset)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  '
              f'train_loss={train_loss:.6f}  val_loss={val_loss:.6f}')

print('Training complete.')

In [ ]:
# Loss curves
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label='Train')
plt.plot(val_losses,   label='Val')
plt.xlabel('Epoch')
plt.ylabel('MAE Loss')
plt.title('Baseline Autoencoder — Training & Validation Loss (MAE)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save checkpoint
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
ckpt_path = CHECKPOINTS_DIR / 'baseline_autoencoder.pth'
torch.save(model.state_dict(), ckpt_path)
print(f'Checkpoint saved to {ckpt_path}')

## 6. Reconstruction Visualisation on Validation Set

In [ ]:
model.eval()
val_batch = next(iter(val_loader)).to(DEVICE)

with torch.no_grad():
    recon_batch = model(val_batch)

n_show = min(8, val_batch.size(0))
fig, axes = plt.subplots(2, n_show, figsize=(2 * n_show, 5))
for i in range(n_show):
    axes[0, i].imshow(denorm(val_batch[i].cpu()))
    axes[0, i].set_title('Input',  fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].imshow(denorm(recon_batch[i].cpu()))
    axes[1, i].set_title('Recon', fontsize=8)
    axes[1, i].axis('off')
fig.suptitle('Validation set — Input vs Reconstruction', fontsize=12)
plt.tight_layout()
plt.show()